In [ ]:
# Import the function from the pipeline module
from pipeline.main import run_cnv_genes_only
from pipeline.config import PATHS

# Call the function
cnv_df = run_cnv_genes_only()

# View the results
display(cnv_df.head())

Running CNV gene processing only...


,chrom,source,feature,start,end,score,strand,frame,gene_id,gene_name,gene_type,transcript_id,transcript_name,transcript_type,tag_list,tag,is_MANE,tss,prom_start,prom_end
306591,chr1,HAVANA,gene,64833223,65067754,.,-,.,ENSG00000162434.15,JAK1,protein_coding,None,None,None,"[ncRNA_host, overlapping_locus]",ncRNA_host|overlapping_locus,False,65067754,65067254,65069754
438368,chr1,HAVANA,gene,145848521,145859836,.,-,.,ENSG00000131788.17,PIAS3,protein_coding,None,None,None,[ncRNA_host],ncRNA_host,False,145859836,145859336,145861836
497813,chr1,HAVANA,gene,154405193,154469450,.,+,.,ENSG00000160712.14,IL6R,protein_coding,None,None,None,[],,False,154405193,154403193,154405693
775646,chr2,HAVANA,gene,9488486,9556732,.,-,.,ENSG00000151694.16,ADAM17,protein_coding,None,None,None,[],,False,9556732,9556232,9558732
781780,chr2,HAVANA,gene,10783378,10837977,.,-,.,ENSG00000143870.14,PDIA6,protein_coding,None,None,None,[ncRNA_host],ncRNA_host,False,10837977,10837477,10839977


In [ ]:
from pipeline.config import PATHS
cnv_df.to_csv(PATHS.cnv_genes, index=False)

In [ ]:
import pandas as pd

def prep_mirna_for_cnv(mirna_df: pd.DataFrame) -> pd.DataFrame:
    """Filters MirGeneDB annotations for CNV intersection."""
    
    # 1. Keep only the genomic loci (precursors)
    clean_df = mirna_df[mirna_df['type'] == 'pre_miRNA'].copy()
    
    # 2. Select only the columns needed for genomic intersection
    clean_df = clean_df[['seqid', 'start', 'end', 'strand', 'ID', 'Alias']]
    
    # 3. Rename columns to match standard BED/PyRanges format (optional but recommended)
    clean_df = clean_df.rename(columns={
        'seqid': 'Chromosome',
        'start': 'Start',
        'end': 'End',
        'strand': 'Strand',
        'ID': 'MirGene_ID',
        'Alias': 'miRBase_Accession'
    })
    
    return clean_df


def parse_attributes(attr_string):
    d = {}
    for item in attr_string.split(";"):
        if "=" in item:
            k, v = item.split("=", 1)
            d[k] = v
    return d

gff = pd.read_csv(
    "data/miRNA/hsa.gff",
    sep="\t",
    comment="#",     # skip header lines
    header=None
)

gff.columns = [
    "seqid", "source", "type", "start", "end",
    "score", "strand", "phase", "attributes"
]

attr_df = gff["attributes"].apply(parse_attributes).apply(pd.Series)

gff = pd.concat([gff.drop(columns="attributes"), attr_df], axis=1)
ready_for_intersection_df = prep_mirna_for_cnv(gff)

In [21]:
ready_for_intersection_df.rename(columns={"Chromosome":"chrom", "MirGene_ID" : "gene_name",
 "miRBase_Accession" : "gene_id", "Start" : "start",
  "End" : "end", "Strand" : "strand"}, inplace=True)


In [22]:
ready_for_intersection_df.to_csv("data/miRNA/cnv_miRNA.csv", index=False)

In [20]:
ready_for_intersection_df

,Chromosome,Start,End,Strand,MirGene_ID,miRBase_Accession
0,chr1,1167124,1167182,+,Hsa-Mir-8-P1a_pre,MI0000342
3,chr1,1167878,1167938,+,Hsa-Mir-8-P2a_pre,MI0000737
6,chr1,1169020,1169076,+,Hsa-Mir-8-P3a_pre,MI0001641
9,chr1,3560710,3560769,-,Hsa-Mir-551-P1_pre,MI0003556
12,chr1,9151692,9151756,-,Hsa-Mir-34-P1_pre,MI0000268
...,...,...,...,...,...,...
1526,chrX,150228016,150228074,+,Hsa-Mir-2114_pre,MI0010633
1529,chrX,151958583,151958651,-,Hsa-Mir-224_pre,MI0000301
1532,chrX,151959639,151959699,-,Hsa-Mir-452-v1_pre,MI0001733
1535,chrX,152392229,152392287,-,Hsa-Mir-105-P2_pre,MI0000111


In [1]:
from pipeline.CNV import process_cnv_directory
from pipeline.config import PATHS
process_cnv_directory(
    cnv_dir=str(PATHS.cnv_dir),
    genes_path=str(PATHS.cnv_genes),
    lncrnas_path=str(PATHS.lncrna_csv),
    mirna_path=str(PATHS.mirna_path),
    elements_path=str(PATHS.regulatory_elements_table),
    ann_path=str(PATHS.cnv_annotations_path),
    out_dir=str(PATHS.cnv_output_dir))

Processing: TCGA-BRCA.4695c0f4-4d30-4604-84b3-81407bf980e8.ascat3.allelic_specific.seg.txt
  -> TCGA-E9-A244-01A
Done -> \home\stavz\masters\gdc\APM\data\CNV_TCGA\CNV_annotated_test


In [7]:
import pandas as pd
import os
os.chdir("../")

from config import PATHS

mirtar = pd.read_csv(PATHS.mirtarbase_input)

In [10]:
mirtar["Target Gene"].nunique()

16981

In [11]:
mirna = pd.read_csv(PATHS.mirna_path)

In [14]:
ccre = pd.read_csv(PATHS.ccre_csv)

In [17]:
ctcf = ccre[ccre["type"].str.contains("CTCF")]
ctcf


,chrom,start,end,cCRE_id,ENCODE_id,type
0,chr1,104896,105048,EH38D4327509,EH38E2776520,"CTCF-only,CTCF-bound"
1,chr1,138866,139134,EH38D4327520,EH38E2776521,"pELS,CTCF-bound"
2,chr1,181289,181639,EH38D4327525,EH38E2776524,"DNase-H3K4me3,CTCF-bound"
3,chr1,267925,268171,EH38D4327544,EH38E2776528,"CTCF-only,CTCF-bound"
4,chr1,586036,586264,EH38D4327554,EH38E2776532,"CTCF-only,CTCF-bound"
...,...,...,...,...,...,...
1063765,chrX,154465495,154465653,EH38D6140892,EH38E3949646,"CTCF-only,CTCF-bound"
1063766,chrX,154472663,154472978,EH38D6140910,EH38E3949652,"CTCF-only,CTCF-bound"
1063769,chrX,154663430,154663743,EH38D6141110,EH38E3949769,"CTCF-only,CTCF-bound"
1063864,chrY,19603850,19604102,EH38D6144188,EH38E3951061,"CTCF-only,CTCF-bound"


In [20]:
genes = pd.read_csv(PATHS.genes_all_features)

In [22]:
genes.columns

Index(['chrom', 'source', 'feature', 'start', 'end', 'score', 'strand',
       'frame', 'gene_id', 'gene_type', 'gene_name', 'level', 'tag',
       'transcript_id', 'transcript_type', 'transcript_name', 'exon_number',
       'exon_id', 'transcript_support_level', 'havana_transcript', 'hgnc_id',
       'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl'],
      dtype='object')